# 00. AICE Associate 왕기초 개념 노트
**처음이라면 이것부터!** 모의고사 풀기 전에 위에서부터 셀을 하나씩 실행하면서 읽어 보세요. (30~40분)

## AICE 시험이 시키는 일, 한 장 요약
```
데이터 읽기 → 눈으로 보기(시각화) → 지저분한 거 청소(전처리)
→ 문제지/정답지 나누기(split) → 크기 맞추기(스케일링)
→ 머신러닝 모델 학습 → 점수 매기기 → 딥러닝 모델 학습 → 그래프로 확인
```
14문항이 거의 이 순서 그대로 나와요. 매번 데이터만 바뀌고 **코드 패턴은 똑같아요.** 그래서 외울 패턴만 외우면 됩니다.

## 1. 파이썬 최소 문법 (이것만 알면 돼요)
| 모양 | 뜻 | 예시 |
|---|---|---|
| `a = 3` | 변수에 저장 | `df = pd.read_csv('a.csv')` |
| `[ ]` | 리스트 (여러 개 묶음) | `['gender', 'region']` |
| `{ }` | 딕셔너리 (짝꿍 묶음) | `{'낮음': 0, '높음': 1}` |
| `함수(값)` | 함수 실행 | `print(3)` |
| `객체.메서드()` | 그 객체가 가진 기능 실행 | `df.head()` |
| `이름=값` | 옵션 지정 | `test_size=0.2` |
| `import A as B` | A를 B라는 별명으로 가져오기 | `import pandas as pd` |

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# 연습용 작은 표 만들기
df = pd.DataFrame({
    'name':  ['A', 'B', 'C', 'D', 'E', 'F'],
    'age':   [25, 32, np.nan, 41, 29, 35],
    'city':  ['서울', '부산', '서울', '대구', '서울', '부산'],
    'score': [80, 95, 70, 300, 88, 76],
    'pass':  [1, 1, 0, 1, 1, 0]
})
df

## 2. pandas 핵심: 표(DataFrame) 다루기
- **DataFrame** = 엑셀 표. **행(row)** 은 가로 줄(사람 한 명), **열(column)** 은 세로 줄(나이, 도시...)
- `df['age']` → 열 하나 꺼내기 (Series라고 불러요)
- `df[['age','city']]` → 열 여러 개 (대괄호 두 겹!)

In [ ]:
print(df.shape)              # (행 개수, 열 개수)
df.info()                    # 열마다 자료형, 결측치 아닌 개수
print(df.describe())         # 숫자 열 요약 (평균, 최소, 최대...)
print(df['city'].value_counts())   # 값마다 몇 개인지
print(df.isnull().sum())     # 열마다 결측치(빈칸) 개수

## 3. 전처리 = 데이터 청소
| 하고 싶은 것 | 코드 |
|---|---|
| 열 삭제 | `df.drop(columns=['name'])` |
| 결측치 채우기 | `df['age'].fillna(df['age'].mean())` (평균) / `.median()` (중앙값) / `.mode()[0]` (최빈값) |
| 결측치 있는 행 삭제 | `df.dropna()` / 특정 열만 보고: `df.dropna(subset=['age'])` |
| 조건으로 행 고르기 | `df[df['score'] <= 100]` |
| 값 바꾸기 | `df['city'].replace('서울', 'Seoul')` / 여러 개: `.map({'서울':0,'부산':1})` |
| 자료형 바꾸기 | `df['age'].astype(int)` / 문자→숫자: `pd.to_numeric(df['x'], errors='coerce')` |
| 중복 행 삭제 | `df.drop_duplicates()` |
| 문자 열을 숫자 열로 펼치기 (원핫) | `pd.get_dummies(df, columns=['city'], drop_first=True)` |

**왜 원핫 인코딩?** 모델은 숫자만 먹어요. '서울/부산/대구'를 `city_서울 = 1/0`, `city_부산 = 1/0` 같은 열로 펼쳐 주는 거예요.

In [ ]:
df1 = df.drop(columns=['name'])                    # 이름은 예측에 쓸모 없음
df1['age'] = df1['age'].fillna(df1['age'].median()) # 빈 나이를 중앙값으로
df1 = df1[df1['score'] <= 100]                      # 300점은 이상치 → 제거
df1 = pd.get_dummies(df1, columns=['city'], drop_first=True)
df1

## 4. 시각화 (seaborn)
| 보고 싶은 것 | 코드 |
|---|---|
| 범주별 개수 | `sns.countplot(data=df, x='city')` |
| 숫자 분포 | `sns.histplot(data=df, x='age')` |
| 범주별 숫자 비교, 이상치 보기 | `sns.boxplot(data=df, x='city', y='score')` |
| 두 숫자 관계 | `sns.scatterplot(data=df, x='age', y='score')` |
| 상관관계 한눈에 | `sns.heatmap(df.corr(numeric_only=True), annot=True)` |

**상관계수**: -1 ~ 1. 1에 가까우면 같이 커지고, -1에 가까우면 반대로 움직이고, 0이면 관계없음. 시험에서는 **절댓값이 큰 것**이 '상관이 높다'는 뜻이에요.

In [ ]:
sns.boxplot(data=df, x='city', y='score')
plt.show()

## 5. 머신러닝 핵심 개념
- **X (특성, feature)**: 문제지. 예측에 쓰는 정보들
- **y (타깃, target, label)**: 정답지. 맞히고 싶은 것
- **train / test**: 공부용 문제(train)와 시험용 문제(test)를 나눠서, 처음 보는 문제로 실력을 확인해요
- **분류 vs 회귀** 구분법

| 타깃이... | 문제 유형 | 예 |
|---|---|---|
| 0/1 두 개 | **이진 분류** | 해지 여부, 연체 여부, 합격 여부 |
| 3개 이상 범주 | **다중 분류** | 등급 A/B/C, 불량 유형 |
| 연속된 숫자 | **회귀** | 가격, 시간, 매출 |

**sklearn 공통 패턴 (모든 모델이 똑같아요!)**
```python
model = 모델이름(옵션)          # 1. 만들기
model.fit(X_train, y_train)    # 2. 공부시키기
pred = model.predict(X_test)   # 3. 예측시키기
```

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

X = df1.drop('pass', axis=1)   # 정답 열을 뺀 나머지 = 문제지
y = df1['pass']                # 정답지

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=42)

model = DecisionTreeClassifier(random_state=42)
model.fit(X_train, y_train)
print(model.predict(X_test), list(y_test))

## 6. 스케일링 (크기 맞추기)
나이(20~40)와 연봉(3000만~1억)을 그대로 넣으면 숫자가 큰 연봉만 중요해 보여요. 그래서 범위를 맞춰 줘요.
- `StandardScaler`: 평균 0, 표준편차 1로
- `MinMaxScaler`: 0 ~ 1 사이로

**규칙: train으로 `fit_transform`, test는 `transform`만.** (시험 문제 내용을 미리 보면 안 되는 것과 같은 원리)

## 7. 평가지표 뜻
| 지표 | 뜻 | 좋은 방향 |
|---|---|---|
| accuracy | 전체 중 맞힌 비율 | 높을수록 |
| precision | '1'이라고 한 것 중 진짜 1 비율 | 높을수록 |
| recall | 진짜 1 중에 찾아낸 비율 | 높을수록 |
| f1-score | precision과 recall의 균형 | 높을수록 |
| MAE | 예측이 평균 몇만큼 틀렸나 (회귀) | **낮을수록** |
| MSE | 틀린 만큼을 제곱해서 평균 (회귀) | **낮을수록** |
| R2 | 얼마나 잘 설명하나, 최대 1 (회귀) | 높을수록 |

## 8. 딥러닝 용어
| 용어 | 뜻 |
|---|---|
| Sequential | 층을 순서대로 쌓는 모델 |
| Dense(64) | 뉴런 64개짜리 층 |
| activation='relu' | 중간층에 쓰는 기본 활성화 함수 |
| Dropout(0.2) | 학습 때 20%를 랜덤으로 꺼서 과적합 방지 |
| epoch | 전체 데이터를 한 바퀴 공부한 횟수 |
| batch_size | 한 번에 몇 개씩 묶어서 공부할지 |
| loss | 틀린 정도. 작아질수록 좋음 |
| val_loss | 검증 데이터 기준 loss. 이게 다시 올라가면 **과적합** |
| EarlyStopping | val_loss가 안 좋아지면 일찍 멈추기 |
| ModelCheckpoint | 가장 좋았던 순간의 모델을 파일로 저장 |

**출력층과 loss 짝꿍 (무조건 암기)**
| 문제 | 출력층 | loss |
|---|---|---|
| 이진 분류 | `Dense(1, activation='sigmoid')` | `'binary_crossentropy'` |
| 다중 분류 | `Dense(클래스수, activation='softmax')` | `'sparse_categorical_crossentropy'` |
| 회귀 | `Dense(1)` | `'mse'` |

---
**이제 01번 모의고사부터 풀어 보세요!** 추천 순서: 01 (이진 분류) → 02 (회귀) → 03 (다중 분류) → 04 (종합, 정답 안 보고)